# Sets

**Topic:** `3.3` &nbsp;|&nbsp; **Module 3: Core Data Structures**

*Unordered collections of unique hashable objects: O(1) membership, set algebra as validation logic, and the traps of orderlessness, unhashable elements and mutation during iteration.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M3)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [03_data_structures/README.md](../README.md)

## 1. Why This Topic Matters

Duplicate detection, fault state, coverage reports and configuration validation all reduce to set operations, and the difference between a quadratic scan and a hash lookup is the difference between an audit that finishes and one that does not. Orderlessness is a real trade, not a defect - provided output is sorted at the boundary.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Construct sets four ways, including set() for the empty set.
2. State the cost of membership on a set versus a list.
3. Apply union, intersection, difference and symmetric difference with correct direction.
4. Express validation with subset, equality and disjoint relations.
5. Deduplicate with a set while preserving order via the seen-set idiom.
6. Explain why set elements must be hashable and why iteration order must not be relied upon.

## 3. Prerequisites

This topic assumes you already have:

- Topic 3.2 Tuples

## 4. Mental Model

**Mental model: a set is a room with named hooks, and each name may hang on exactly one hook.** Inserting an already-present name finds its hook and does nothing; there is no 'first' or 'last' hook, so no order exists to remember; and a name you cannot hash is a name the doorkeeper cannot place, so it never gets in.

| Question | Answer | Consequence |
| --- | --- | --- |
| Duplicates? | collapsed by construction | `set(xs)` dedupes in O(n) |
| Order? | none stored | `sorted(s)` whenever output matters |
| `s[0]`? | TypeError - no indexing | iterate, test membership, or use algebra |
| `x in s` cost? | O(1) average | the reason sets exist |
| Lists as elements? | TypeError - unhashable | convert to tuples |
| `{}`? | an empty dict | empty set is `set()` |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Set | An unordered collection of unique, hashable objects. |
| Membership test | `x in s` - an O(1) average hash lookup. |
| Hashable | An object with a stable hash; required for set elements. |
| Unordered | No indexing or insertion order; iteration order is arbitrary. |
| Union | `a \| b` - elements present in either set. |
| Intersection | `a & b` - elements present in both sets. |
| Difference | `a - b` - elements of a absent from b. |
| Symmetric difference | `a ^ b` - elements in exactly one of the sets. |
| Subset / superset | `a <= b` - every element of a is in b (relations as booleans). |
| frozenset | The immutable, hashable sibling of set - usable as a set element or key. |

## 6. Conceptual Explanation

A set is an **unordered collection of unique, hashable objects**. Uniqueness is not a feature applied on top - it is the type's identity: putting the same value into a set twice leaves one element, because a set has exactly one slot per distinct hash. Unordered means there is no `set[0]`, no indexing, no slicing: the only sensible ways to read a set are iteration, membership tests and set algebra.

The payoff for dropping order is speed. Membership `x in s` is O(1) on average - the same hash lookup a dict performs - where the same test against a list is O(n). That single difference decides which type belongs in a hot loop: a seen-set turns a quadratic duplicate scan into a linear one.

In [ ]:
import time

values = list(range(20000))
member_set = set(values)

start = time.perf_counter()
hits_list = sum(1 for v in values if v in values)
list_s = time.perf_counter() - start

start = time.perf_counter()
hits_set = sum(1 for v in values if v in member_set)
set_s = time.perf_counter() - start

print(f'list scan {list_s:.4f}s vs set scan {set_s:.4f}s')
print(f'set is {list_s / max(set_s, 1e-9):.0f}x faster here')

Two rules follow from hashability. First, every element must be hashable - lists and dicts are banned, tuples and strings are fine. Second, because the set decides placement by hash, it must not be modified while being iterated; like a dictionary, growing a set during iteration raises RuntimeError.

Sets also carry *relations*: subset, superset and disjoint describe how two sets overlap, and those relations power the whole family of set-algebra operations - union, intersection, difference and symmetric difference - each returning a new set rather than mutating the operands.

## 7. Formal Theory

A set implements the mathematical finite set over hashable elements. Membership, insertion and deletion are O(1) average (O(n) worst case under hash collisions); every set-algebra operation is linear in the size of the operands:

```text
A | B = union   A & B = intersection   A - B = difference   A ^ B = symmetric difference
```

In [ ]:
crew = {'ada', 'grace', 'alan'}
on_call = {'grace', 'linus'}

print('union        :', crew | on_call)
print('intersection :', crew & on_call)
print('crew - on_call:', crew - on_call)
print('symmetric    :', crew ^ on_call)

| Expression | Meaning | Returns |
| --- | --- | --- |
| `a \| b` | elements in either | new set |
| `a & b` | elements in both | new set |
| `a - b` | elements of a missing from b | new set |
| `a ^ b` | elements in exactly one | new set |
| `a <= b` | a is a subset of b | bool |
| `a.isdisjoint(b)` | a and b share nothing | bool |

The in-place operators exist too: `a |= b`, `a &= b`, `a -= b` and `a ^= b` mutate the left operand and return it, exactly like list methods returning None. The distinction matters in shared code - `result = a | b` preserves `a`, while `a |= b` rebinds and mutates `a`, so the caller who passed `a` sees a different set afterward.

Construction has four forms, and the set-literal `{...}` is only one of them - `{}` is an empty **dict**, not an empty set, because braces are already claimed by dict display. `set(iterable)` converts, `set()` builds empty, and `{x for x in ...}` is a set comprehension.

In [ ]:
empty_dict = {}                       # a dict, not a set!
empty_set = set()                     # the empty set
literal = {1, 2, 3}
from_comp = {v * 2 for v in range(3)}
converted = set(['a', 'a', 'b'])      # duplicates collapse

print('dict :', type(empty_dict).__name__)
print('set  :', type(empty_set).__name__, empty_set)
print('comp :', from_comp)
print('dedup:', converted)

## 8. Syntax

Construction, membership, and the mutating algebra - each with its non-mutating twin.

In [ ]:
tags = {'fault', 'warn'}               # set literal
tags.add('info')                       # one element, in place
tags.update(['fault', 'debug'])        # many; 'fault' is deduped
tags.discard('debug')                  # missing element is OK
tags.remove('info')                    # missing element raises

print('tags  :', tags)
print('member:', 'warn' in tags)       # O(1) membership

unioned = tags | {'calibrated'}        # NEW set; tags unchanged
print('original still:', tags)
print('union         :', unioned)

The anti-pattern - building a unique list by hand, which is quadratic:

```text
seen = []
for value in stream:
    if value not in seen:   # O(n) per test -> O(n^2) overall
        seen.append(value)

seen = set()
for value in stream:
    seen.add(value)         # O(1) per insert -> O(n) overall
# order needed later? sort() it, or keep a parallel list.
```

> **`{}` creates a dict**
>
> The empty set must be written `set()`. A bare `{}` is an empty dict, and the mistake hides until `{}.add(...)` raises AttributeError - which is why type() in tests catches it faster than reading the code.

## 9. Basic Examples

**Example 1 - deduplicate a stream in one line.**

In [ ]:
readings = [0.4, 0.9, 0.4, 0.7, 0.9]
unique = set(readings)

print('original :', readings, ' len', len(readings))
print('unique   :', sorted(unique), ' len', len(unique))
print('duplicates removed:', len(readings) - len(unique))

**Example 2 - membership as a fast lookup table.**

In [ ]:
fault_codes = {'E01', 'E02', 'E17'}

for code in ('E01', 'E09'):
    print(f'{code} known?', code in fault_codes)

try:
    fault_codes.add('E01')          # already present: no change
    print('add is idempotent:', fault_codes)
except Exception as exc:
    print('unexpected:', exc)

**Example 3 - a set comprehension for derived collections.**

In [ ]:
log = ['BOOT', 'WARN', 'BOOT', 'CAL', 'WARN']
events = {line.split()[0] for line in log}
print('distinct event kinds:', sorted(events))

first_letters = {name[0] for name in ('ada', 'alan', 'grace')}
print('initials:', sorted(first_letters))

## 10. Intermediate Examples

**The seen-set idiom.** The single most valuable set pattern in Python: converting a repeated `not in list` test into an O(1) hash lookup, usually while preserving first-seen order in a parallel list.

In [ ]:
def first_seen_order(values):
    """Unique values in first-seen order."""
    seen = set()
    ordered = []
    for value in values:
        if value not in seen:
            seen.add(value)
            ordered.append(value)
    return ordered

print(first_seen_order(['b', 'a', 'b', 'c', 'a']))

**Relations between sets** answer questions list membership cannot: "is every subscriber also a crew member?", "do these two fault sets overlap at all?"

In [ ]:
crew = {'ada', 'grace', 'alan'}
pilots = {'grace'}
operators = {'linus', 'ada'}

print('pilots subset of crew?', pilots <= crew)
print('crew covers operators?', crew >= operators)
print('disjoint with operators?', crew.isdisjoint(operators))
print('shared:', sorted(crew & operators))

**Frozen sets complete the lattice.** Just as tuple is list's immutable sibling, `frozenset` is set's: hashable, so a frozenset may live inside a set or serve as a dict key.

In [ ]:
roles = {frozenset({'pilot'}), frozenset({'pilot', 'engineer'})}
print('hashable nested sets:', roles)

try:
    { {'a'} }
except TypeError as exc:
    print('mutable set rejected:', exc)

## 11. Advanced Examples

**Ordering is not just missing - it is deliberately arbitrary.** A set's iteration order depends on hash values and insertion history, so identical sets built in different orders may iterate differently. Any code that needs reproducible output must sort explicitly.

In [ ]:
a = set(['x', 'y', 'z'])
b = set(['z', 'y', 'x'])

print('equal sets?', a == b)              # True - equality is content
print('same order?', list(a) == list(b))  # NOT guaranteed
print('reproducible:', sorted(a))         # sort when order matters

**Set algebra as validation.** 'Exactly these keys were supplied' is one symmetric difference away: `supplied ^ expected` is empty if and only if the two sets match, and the difference names precisely what is extra or missing.

In [ ]:
expected = {'host', 'port', 'timeout'}
supplied = {'host', 'port', 'retries'}

extra = supplied - expected
missing = expected - supplied
print('extra   :', sorted(extra))
print('missing :', sorted(missing))
print('exact match?', supplied == expected)
print('symmetric diff:', sorted(supplied ^ expected))

> **do not mutate a set while iterating it**
>
> Adding or discarding elements during `for x in s` raises RuntimeError: Set changed size during iteration. Iterate a copy - `for x in set(s)` or `for x in list(s)` - or rebuild the result with a comprehension.

In [ ]:
faults = {'E01', 'E02', 'E03'}
try:
    for code in faults:
        if code == 'E02':
            faults.add('E99')
except RuntimeError as exc:
    print('caught:', exc)

cleared = {c for c in faults if c != 'E02'}   # the clean way
print('filtered:', sorted(cleared))

> **counting with Counter**
>
> When uniqueness gives way to counting, `collections.Counter` is a dict subclass built for exactly that. Reach for a set when you need *presence*; reach for Counter when you need *how many*.

## 12. Code Walkthrough

**A duplicate detector for incoming telemetry frames.** Frames arrive as `(channel, seq)` tuples; replayed frames must be dropped without an O(n) scan, and the report says how many were rejected.

In [ ]:
def dedupe_frames(frames):
    """Drop replayed (channel, seq) frames, preserving first-seen order."""
    seen = set()
    fresh = []
    for frame in frames:
        if frame in seen:
            continue                    # replay - already logged
        seen.add(frame)
        fresh.append(frame)
    rejected = len(frames) - len(fresh)
    return fresh, rejected

stream = [('imu', 1), ('imu', 2), ('imu', 1), ('gps', 1)]
clean, rejected = dedupe_frames(stream)
print('fresh    :', clean)
print('rejected :', rejected)

- `seen = set()` is the O(1) membership table; tuples hash fine, so frames drop straight in.
- `continue` on a hit keeps the happy path flush left - the membership test is the only branch.
- The parallel `fresh` list preserves arrival order, which the set itself cannot do.
- Reporting `rejected` as a count avoids keeping the discarded frames in memory.

## 13. Line-by-Line Explanation

1. `seen = set()` - an empty membership table, O(1) per lookup from the first frame onward.
2. `for frame in frames:` - one pass over the stream; frames are tuples, so they are hashable.
3. `if frame in seen: continue` - the whole duplicate test; with a list this line would be O(n).
4. `seen.add(frame)` - registers the frame BEFORE appending, so a second copy later hits the test.
5. `fresh.append(frame)` - order-preserving output the set cannot provide on its own.
6. `rejected = len(frames) - len(fresh)` - a derived count, cheaper than storing rejects and consistent by construction.

## 14. Common Mistakes

**Mistake 1 - writing `{}` for the empty set.**

```text
tags = {}            # dict!
tags.add('x')        # AttributeError: 'dict' object has no attribute 'add'
tags = set()         # the empty set
```

**Mistake 2 - expecting a set to remember insertion order.**

```text
s = {'c', 'a', 'b'}
print(list(s))       # some order - NOT insertion order, not sorted
print(sorted(s))     # ['a', 'b', 'c'] - explicit, reproducible
```

**Mistake 3 - putting unhashable elements in a set.**

```text
bad = {[1, 2]}       # TypeError: unhashable type: 'list'
good = {(1, 2)}      # tuples hash fine
also = {frozenset({1, 2})}
```

**Mistake 4 - mutating a set during iteration.**

```text
s = {1, 2, 3}
for x in s:
    s.add(x + 10)    # RuntimeError: Set changed size during iteration

for x in set(s):     # iterate a snapshot instead
    ...
```

**Mistake 5 - using `remove` on a possibly-absent element.**

```text
s = {'a'}
s.remove('b')        # KeyError - remove is strict
s.discard('b')       # no error - discard is forgiving
```

## 15. Debugging Techniques

When a set seems to lose or invent elements, print sorted() so the contents are stable, and print the *type* of the container itself - `{}` vs `set()` and list-vs-set confusions are diagnosed in one line.

In [ ]:
container = {}
print('type:', type(container).__name__)

tags = set(['a', 'a', 'b'])
print('deduped contents:', sorted(tags))
print('expected 2, got :', len(tags))

unhashable = None
try:
    {unhashable or [1]}
except TypeError as exc:
    print('hash failure:', exc)

When two sets 'disagree', compare them with the algebra itself - symmetric difference names the disagreement, and equality is one line.

In [ ]:
expected = {'host', 'port'}
actual = {'host', 'timeout'}

print('equal?', expected == actual)
print('only expected:', sorted(expected - actual))
print('only actual  :', sorted(actual - expected))
print('both         :', sorted(expected & actual))

> **Hash failures name the culprit**
>
> TypeError: unhashable type: 'dict' always names the inner type - walk your structure until you find it. The fix is almost always a tuple conversion or a frozenset.

## 16. Best Practices

- Use a set for membership tests that repeat over a collection - it is the difference between O(n^2) and O(n).
- Write `set()` for the empty set; treat `{}` as a dict always.
- Preserve order with a parallel list when order matters; never rely on set iteration order for output.
- Sort (`sorted(s)`) before printing, logging or snapshot testing a set.
- `discard` for optional removals, `remove` when absence is a real error worth raising.
- Validate 'exactly these keys' with `==` on sets or a symmetric difference, not a chain of `in` checks.
- Use frozenset when a set must be a dict key or set element.
- Rebuild with comprehensions instead of mutating during iteration.
- Convert to a set for deduplication only when order is irrelevant - otherwise use the seen-set idiom from the walkthrough.

## 17. Performance Considerations

A set is a hash table: average O(1) membership, insertion and deletion, O(n) worst case only under hash collisions. Every algebra operation walks at least one operand, so their cost is linear in the input sizes - and the smaller operand should usually be the one walked.

| Pattern | Cost | Preferred alternative |
| --- | --- | --- |
| `x in list` inside a loop | O(n) per test - quadratic total | `seen = set(list)` once, then O(1) |
| manual dedup with `if v not in out` | O(n^2) | `set(values)` or the seen-set idiom |
| `for x in a: if x in b` | O(len(a) * len(b)) | `a & b` runs in O(len(a) + len(b)) |
| sorted output from a set | O(n log n) | unavoidable - order is not stored |
| `set(list)` construction | O(n) expected | the right default for dedup |

In [ ]:
import time

n = 30000
values = list(range(n))

start = time.perf_counter()
naive = []
for v in values:
    if v not in naive:          # O(n) scan per element
        naive.append(v)
naive_s = time.perf_counter() - start

start = time.perf_counter()
seen = set()
fast = []
for v in values:
    if v not in seen:           # O(1) lookup per element
        seen.add(v)
        fast.append(v)
fast_s = time.perf_counter() - start

print(f'naive {naive_s:.4f}s vs seen-set {fast_s:.4f}s')
print(f'seen-set is {naive_s / max(fast_s, 1e-9):.1f}x faster')

The quadratic term is what the table means in practice: doubling n quadruples the naive loop's work while doubling the seen-set's. At 30000 elements the gap is already orders of magnitude - and at a million, the naive version would not finish.

## 18. Pythonic Approaches

Sets make whole classes of checkable intent expressible in one expression: membership relations, exact-set validation and deduplication - each replacing a loop the reader would have to audit by eye.

In [ ]:
required = {'host', 'port'}
supplied = {'host', 'port', 'timeout'}

# not pythonic: a chain of membership checks
ok = 'host' in supplied and 'port' in supplied

# pythonic: the relation IS the check
ok = required <= supplied
print('all required supplied?', ok)
print('unexpected keys      :', sorted(supplied - required))

- `set(xs)` to deduplicate; pair it with `sorted()` when output must be stable.
- `required <= supplied` replaces chains of `in` when validating presence.
- `a ^ b == set()` (or `a == b`) for exact-set equality checks.
- `set.isdisjoint` reads better than `not (a & b)` - and short-circuits.
- Set comprehensions `{f(x) for x in xs}` dedupe as they map.
- `dict.fromkeys(xs).keys()` preserves first-seen order when you need dedup WITH order (a dict view, not a set).
- Complement loops with `seen` sets rather than re-scanning the accumulator.

## 19. Robotics Connection

A rover's fault supervisor tracks active fault codes as a set: duplicates are meaningless (a fault is either present or not), the supervisor asks 'is E17 active?' thousands of times a second, and clearing is one discard per code. A list would turn each of those questions into a scan.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
active = set()

if robot.is_low_battery(threshold_pct=100.0):
    active.add('E_BATT')
active.add('E_BATT')          # repeat report: still one entry

print('active faults:', sorted(active))
print('E_BATT active?', 'E_BATT' in active)   # the hot-path query
print('cleared:')
active.discard('E_BATT')
print(' ', sorted(active))

The engineering rule: presence questions get sets, order questions get lists. A fault *log* is a list (events in time order); a fault *state* is a set (codes present now). Confusing the two either costs quadratic scans or loses the timeline - and the supervisor's `isdisjoint` check against a set of critical codes is the kind of one-line safety test sets make possible.

## 20. Engineering Example

Configuration validation is where set relations replace boilerplate: compare supplied keys against required keys and report both unexpected and missing entries from one symmetric difference.

In [ ]:
def validate_config(supplied, required):
    """Return (unexpected, missing) key lists; empty means valid."""
    supplied, required = set(supplied), set(required)
    unexpected = sorted(supplied - required)
    missing = sorted(required - supplied)
    return unexpected, missing

ok_u, ok_m = validate_config(['host', 'port'], {'host', 'port'})
bad_u, bad_m = validate_config(['host', 'colour'], {'host', 'port'})
print('valid case:', (ok_u, ok_m))
print('unexpected:', bad_u, ' missing:', bad_m)

Sorting inside the function makes the report reproducible across runs - a set's iteration order is not - and converting both sides with `set(...)` means callers may pass lists, tuples or sets interchangeably. The whole validation is four lines and has no loop to get wrong.

## 21. Guided Practice

1. Build a set three ways - literal, `set(list)` and a comprehension - and print `type()` and `len()` for each to see duplicates collapse.
2. Prove `{}` is a dict by calling `.add()` on it and reading the AttributeError, then use `set()` correctly.
3. Take two small sets and print union, intersection, difference and symmetric difference with `sorted()`.
4. Reproduce the RuntimeError from adding during iteration, then fix it by iterating `set(s)`.
5. Time a duplicate scan against a seen-set for 20000 elements and record the ratio.

In [ ]:
tags = {'fault', 'warn', 'fault'}
from_list = set(['a', 'b', 'a'])
from_comp = {n * n for n in range(4)}
print('literal:', sorted(tags), ' from_list:', sorted(from_list), ' comp:', sorted(from_comp))

try:
    {}
    {}.add('x')
except AttributeError as exc:
    print('braces are a dict:', exc)

a, b = {1, 2, 3}, {2, 3, 4}
print('union:', sorted(a | b), ' inter:', sorted(a & b))
print('a - b:', sorted(a - b), ' sym:', sorted(a ^ b))

s = {1, 2}
try:
    for x in s:
        s.add(x + 10)
except RuntimeError as exc:
    print('mutation guard:', exc)

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** What does `type({})` produce, and why does it matter?
- **Q1.** What is the average cost of `x in my_set` versus `x in my_list`?
- **Q1.** A supervisor must not modify a fault set while reporting on it. What is the correct pattern?
- **Q1.** Why is frozenset the correct key for grouping anagrams by letter-set?

## 23. Summary

A set is an unordered bag of unique hashable objects, and every choice it makes follows from that: uniqueness collapses duplicates the moment they arrive, order is not stored so output must be sorted, and hashability bans lists and dicts as elements. The payoff is O(1) average membership - the difference between a quadratic duplicate scan and a linear one.

The algebra is the second half of the topic. Union, intersection, difference and symmetric difference each build a new set, while the in-place `|=`, `&=`, `-=` and `^=` mutate the left operand exactly like list methods returning None. Relations - subset, superset, disjoint - turn validation questions (`required <= supplied`) into single expressions that cannot be mis-ordered by hand.

In practice: presence questions are set questions. A seen-set upgrades any repeated membership loop from O(n^2) to O(n); `set(xs)` deduplicates whenever order does not matter; frozenset completes the lattice for nested keys. The failure modes - `{}` as dict, mutation during iteration, unhashable elements - are all loud, and each one names its own fix in the error message.

## 24. Key Takeaways

- Sets are unordered, unique, and hashable-only; `{}` is a dict, `set()` is the empty set.
- `x in s` is O(1) average - the tool that removes quadratic loops.
- `set(xs)` deduplicates in O(n); pair with `sorted()` for stable output.
- Algebra operators return new sets; `|=`, `&=`, `-=` and `^=` mutate in place.
- Use `<=`, `>=`, `isdisjoint` and `==` to express relations instead of loop chains.
- `discard` ignores missing elements; `remove` raises KeyError.
- Never mutate a set while iterating it - iterate a snapshot or rebuild with a comprehension.
- frozenset is set's hashable form, for sets inside sets and dict keys.

## 25. Further Exploration

- Read the set-method table in the docs: which methods return new sets and which return None?
- Compare `set.add` against `list.append` timing for 10^5 inserts, then for membership probes.
- Investigate `collections.Counter.most_common` as the counting cousin of the seen-set.
- Explore why dict preserves insertion order in CPython while set does not (PEP 468 and the dict's combined table).
- Prove `a ^ b == (a | b) - (a & b)` with two random sets and then on paper.